In [3]:
%pip -q install faiss-cpu rank-bm25 sentence-transformers pandas numpy tqdm

In [5]:
from google.colab import drive

drive.mount("/content/drive")

ValueError: Mountpoint must not already contain files

In [6]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import hashlib
import json
import os
import re
import time
import shutil

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

PROJECT_DIR = Path('/content/drive/MyDrive/NLP Project')
EVAL_DIR = PROJECT_DIR / 'data/evaluation'
RETRIEVAL_DIR = PROJECT_DIR / 'data/retrieval_results/evaluation_95'
OUTPUT_ROOT = PROJECT_DIR / 'data/answer_generation/experimental_study'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

QUESTIONS_FILE = EVAL_DIR / 'evaluation_questions.csv'
RETRIEVAL_FILE = RETRIEVAL_DIR / 'retrieval_top10_all_methods.csv'
TIMINGS_FILE = RETRIEVAL_DIR / 'retrieval_timings.csv'

SELECTED_STRATEGY = 'section_aware'  # Keep the same corpus/chunks for B–E.
SYSTEM_METHODS = {
    'A': None,
    'B': 'bm25',
    'C': 'dense_faiss',
    'D': 'hybrid_reranked',
    'E': 'hybrid_rrf',
}
SYSTEM_LABELS = {
    'A': 'No retrieval',
    'B': 'Lexical RAG (BM25)',
    'C': 'Dense RAG (FAISS)',
    'D': 'Full hybrid RAG + Cross-Encoder',
    'E': 'Hybrid RRF without Cross-Encoder',
}

API_PROVIDER = 'openai'
OPENAI_CHAT_URL = 'https://api.openai.com/v1/chat/completions'
COLAB_SECRET_NAME = 'OPENAI_API_KEY'
LLM_MODEL = 'gpt-4.1-mini'  # Change only before the first run; model must match the judge study design

CONTEXT_K = 8
MAX_PASSAGE_CHARS = 1800
MAX_OUTPUT_TOKENS = 1600
TEMPERATURE = 0
REQUEST_TIMEOUT = 120
MAX_RETRIES = 4
PROMPT_VERSION = 'controlled-abcd-e-v1-openai-strict'

# OpenAI Structured Outputs schema for answer generation. All keys required.
GEN_RESPONSE_FORMAT = {
    'type': 'json_schema',
    'json_schema': {
        'name': 'research_qa_answer',
        'strict': True,
        'schema': {
            'type': 'object',
            'properties': {
                'status': {'type': 'string', 'enum': ['answered', 'insufficient_evidence']},
                'answer': {'type': 'string'},
                'citations': {'type': 'array', 'items': {'type': 'string'}},
            },
            'required': ['status', 'answer', 'citations'],
            'additionalProperties': False,
        },
    },
}

if not 1 <= CONTEXT_K <= 10:
    raise ValueError('CONTEXT_K must be 1–10 (Notebook 09 saves top 10).')
if MAX_PASSAGE_CHARS <= 0 or MAX_OUTPUT_TOKENS <= 0:
    raise ValueError('Passage and generation output limits must be positive.')
for path in (QUESTIONS_FILE, RETRIEVAL_FILE):
    if not path.is_file():
        raise FileNotFoundError(f'Missing Notebook 09 input: {path}')

def file_sha256(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        while True:
            block = stream.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

# Provider/model/input/format changes create distinct generation runs.
RUN_SETTINGS = {
    'api_provider': API_PROVIDER,
    'api_endpoint': OPENAI_CHAT_URL,
    'model': LLM_MODEL,
    'chunking_strategy': SELECTED_STRATEGY,
    'systems': SYSTEM_METHODS,
    'context_k': CONTEXT_K,
    'max_passage_chars': MAX_PASSAGE_CHARS,
    'max_output_tokens': MAX_OUTPUT_TOKENS,
    'temperature': TEMPERATURE,
    'prompt_version': PROMPT_VERSION,
    'generation_response_format': GEN_RESPONSE_FORMAT,
    'question_file_sha256': file_sha256(QUESTIONS_FILE),
    'retrieval_file_sha256': file_sha256(RETRIEVAL_FILE),
}
RUN_ID = hashlib.sha256(
    json.dumps(RUN_SETTINGS, sort_keys=True).encode('utf-8')
).hexdigest()[:12]
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FILE = OUTPUT_DIR / 'answers_all_systems.csv'
CONFIG_FILE = OUTPUT_DIR / 'run_config.json'

# Never silently overwrite config metadata with different settings.
if CONFIG_FILE.exists():
    recorded = json.loads(CONFIG_FILE.read_text(encoding='utf-8'))
    if recorded != RUN_SETTINGS:
        raise ValueError('Existing run_config.json differs from the current settings.')
else:
    CONFIG_FILE.write_text(json.dumps(RUN_SETTINGS, indent=2), encoding='utf-8')

print('Provider:', API_PROVIDER, '| Model:', LLM_MODEL)
print('Systems:', ', '.join(SYSTEM_METHODS), '| Strategy:', SELECTED_STRATEGY)
print('Input corpus: 95 questions | Target subset: 40 questions | Planned A–E answers: 200')
print('Generation RUN_ID:', RUN_ID)
print('Output folder:', OUTPUT_DIR)
print('OpenAI checkpoint is separated from prior Groq experiments.')

FileNotFoundError: Missing Notebook 09 input: /content/drive/MyDrive/NLP Project/data/evaluation/evaluation_questions.csv

Load and validate your research questions

In [10]:
# 1. LOAD EVALUATION DATA

EXPECTED_QUESTIONS = 95

questions = pd.read_csv(
    QUESTIONS_FILE,
    dtype={"question_id": "string"}
)

gold = pd.read_csv(
    GOLD_FILE,
    dtype={
        "question_id": "string",
        "article_id": "string"
    }
)


# 2. VALIDATE REQUIRED COLUMNS

question_columns = {
    "question_id",
    "question",
    "question_type",
    "reference_answer",
    "answerable"
}

gold_columns = {
    "question_id",
    "article_id"
}

missing_questions = question_columns - set(questions.columns)
missing_gold = gold_columns - set(gold.columns)

if missing_questions:
    raise ValueError(
        f"Missing question columns: {sorted(missing_questions)}"
    )

if missing_gold:
    raise ValueError(
        f"Missing gold evidence columns: {sorted(missing_gold)}"
    )


# 3. CLEAN TEXT AND IDENTIFIERS

text_columns = [
    "question_id",
    "question",
    "question_type",
    "reference_answer"
]

for column in text_columns:
    questions[column] = (
        questions[column]
        .astype("string")
        .str.strip()
    )

for column in ["question_id", "article_id"]:
    gold[column] = (
        gold[column]
        .astype("string")
        .str.strip()
    )


# 4. CONVERT ANSWERABLE VALUES TO BOOLEAN

def parse_answerable(value):
    """Convert CSV answerability values to True or False."""

    if pd.isna(value):
        raise ValueError("Missing answerable value.")

    value = str(value).strip().lower()

    if value in {"true", "1", "yes"}:
        return True

    if value in {"false", "0", "no"}:
        return False

    raise ValueError(
        f"Invalid answerable value: {value}"
    )


questions["answerable"] = (
    questions["answerable"].map(parse_answerable)
)


# 5. VALIDATE QUESTION COUNT AND IDs

if len(questions) != EXPECTED_QUESTIONS:
    raise ValueError(
        f"Expected {EXPECTED_QUESTIONS} questions, "
        f"found {len(questions)}."
    )

if (
    questions["question_id"].isna().any()
    or questions["question_id"].eq("").any()
):
    raise ValueError("Missing or empty question IDs detected.")

if questions["question_id"].duplicated().any():

    duplicates = questions.loc[
        questions["question_id"].duplicated(keep=False),
        "question_id"
    ].tolist()

    raise ValueError(
        f"Duplicate question IDs: {duplicates}"
    )

if (
    questions["question"].isna().any()
    or questions["question"].eq("").any()
):
    raise ValueError("Missing or empty questions detected.")


# 6. VALIDATE QUESTION CATEGORIES

required_categories = [
    "direct_factual",
    "conceptual_explanatory",
    "multi_paper_comparison",
    "unanswerable_insufficient_evidence"
]

if (
    questions["question_type"].isna().any()
    or questions["question_type"].eq("").any()
):
    raise ValueError(
        "Missing or empty question categories detected."
    )

category_counts = (
    questions["question_type"].value_counts()
)

unknown_categories = (
    set(category_counts.index)
    - set(required_categories)
)

if unknown_categories:
    raise ValueError(
        f"Unexpected question categories: {unknown_categories}"
    )

# Require at least 20 questions per category
for category in required_categories:

    count = int(category_counts.get(category, 0))

    if count < 20:
        raise ValueError(
            f"{category}: only {count} questions. "
            "Minimum required is 20."
        )


# 7. VALIDATE ANSWERABILITY

expected_answerable = (
    questions["question_type"]
    != "unanswerable_insufficient_evidence"
)

if not (
    questions["answerable"] == expected_answerable
).all():
    raise ValueError(
        "Question categories and answerable labels disagree."
    )

answerable_questions = questions.loc[
    questions["answerable"]
].copy()

unanswerable_questions = questions.loc[
    ~questions["answerable"]
].copy()

# Every answerable question needs a reference answer
if (
    answerable_questions["reference_answer"].isna().any()
    or answerable_questions["reference_answer"].eq("").any()
):
    raise ValueError(
        "Some answerable questions have missing reference answers."
    )


# 8. VALIDATE GOLD IDs AND FILTER OBSOLETE EVIDENCE

# Validate identifiers before filtering any evidence
if (
    gold["question_id"].isna().any()
    or gold["question_id"].eq("").any()
):
    raise ValueError(
        "Gold evidence contains missing or empty question IDs."
    )

if (
    gold["article_id"].isna().any()
    or gold["article_id"].eq("").any()
):
    raise ValueError(
        "Gold evidence contains missing or empty article IDs."
    )

question_ids = set(questions["question_id"])

removed_evidence = gold.loc[
    ~gold["question_id"].isin(question_ids)
].copy()

if not removed_evidence.empty:

    removed_ids = sorted(
        removed_evidence["question_id"]
        .unique()
        .tolist()
    )

    print("Obsolete evidence question IDs:", removed_ids)

    print(
        f"Excluding {len(removed_evidence)} "
        "obsolete evidence records."
    )

    # Filter in memory; original CSV remains unchanged
    gold = gold.loc[
        gold["question_id"].isin(question_ids)
    ].copy()


# 9. VALIDATE PDF PAGE NUMBERS

if "pdf_page_number" not in gold.columns:

    if "page_number" in gold.columns:
        gold = gold.rename(
            columns={"page_number": "pdf_page_number"}
        )

    else:
        raise ValueError(
            "Gold evidence requires pdf_page_number."
        )

page_values = pd.to_numeric(
    gold["pdf_page_number"],
    errors="coerce"
)

if page_values.isna().any():
    raise ValueError(
        "Gold evidence contains invalid PDF page numbers."
    )

if (page_values < 1).any():
    raise ValueError(
        "PDF page numbers must start from 1."
    )

if (page_values % 1 != 0).any():
    raise ValueError(
        "PDF page numbers must be whole numbers."
    )

gold["pdf_page_number"] = page_values.astype(int)


# 10. CHECK DUPLICATE GOLD EVIDENCE

# Check duplicate question/article/page combinations
# without removing potentially useful evidence.

duplicate_evidence = gold.duplicated(
    subset=[
        "question_id",
        "article_id",
        "pdf_page_number"
    ],
    keep=False
)

if duplicate_evidence.any():

    print(
        "\nWarning:",
        int(duplicate_evidence.sum()),
        "records share the same question, article and page."
    )

    print(
        "Review these records if multiple evidence "
        "passages on the same page are not intentional."
    )


# 11. VERIFY GOLD EVIDENCE COVERAGE

answerable_ids = set(
    answerable_questions["question_id"]
)

unanswerable_ids = set(
    unanswerable_questions["question_id"]
)

gold_ids = set(gold["question_id"])

# Unanswerable questions must not have gold evidence
invalid_evidence = gold_ids & unanswerable_ids

if invalid_evidence:
    raise ValueError(
        "Unanswerable questions contain gold evidence: "
        f"{sorted(invalid_evidence)}"
    )

# Every answerable question must have gold evidence
missing_evidence = answerable_ids - gold_ids

if missing_evidence:
    raise ValueError(
        "Answerable questions missing evidence: "
        f"{sorted(missing_evidence)}"
    )


# 12. VALIDATE MULTI-PAPER QUESTIONS

multi_questions = set(
    questions.loc[
        questions["question_type"]
        == "multi_paper_comparison",
        "question_id"
    ]
)

article_counts = (
    gold.groupby("question_id")["article_id"]
    .nunique()
)

for qid in multi_questions:

    count = int(article_counts.get(qid, 0))

    if count < 2:
        raise ValueError(
            f"{qid} has only {count} supporting articles. "
            "Multi-paper questions require at least 2."
        )


# 13. DISPLAY VALIDATION SUMMARY

print("\n" + "=" * 60)
print("EVALUATION DATASET VALIDATION")
print("=" * 60)

print("\nDataset validation: PASSED")
print("\nTotal questions:", len(questions))

print("\nQuestion category distribution:")

print(
    questions["question_type"]
    .value_counts()
    .reindex(required_categories, fill_value=0)
)

print("\nAnswerable questions:", len(answerable_questions))
print("Unanswerable questions:", len(unanswerable_questions))

print("\nGold evidence records:", len(gold))

print(
    "Distinct supporting articles:",
    gold["article_id"].nunique()
)

print(
    "Questions with gold evidence:",
    gold["question_id"].nunique()
)

print("\nAll required validation checks passed.")

display(questions.head())
display(gold.head())



EVALUATION DATASET VALIDATION

Dataset validation: PASSED

Total questions: 95

Question category distribution:
question_type
direct_factual                        22
conceptual_explanatory                24
multi_paper_comparison                24
unanswerable_insufficient_evidence    25
Name: count, dtype: Int64

Answerable questions: 70
Unanswerable questions: 25

Gold evidence records: 221
Distinct supporting articles: 83
Questions with gold evidence: 70

All required validation checks passed.


,question_id,question,question_type,reference_answer,answerable
0,Q001,Which two ensemble methods were evaluated in t...,direct_factual,AdaBoost and majority voting.,True
1,Q002,Which two drift-detection procedures were inco...,direct_factual,Early Drift Detection Method (EDDM) and ADapti...,True
2,Q003,Which three feature-selection techniques are i...,direct_factual,"Pearson correlation, information gain (IG), an...",True
3,Q004,Which three datasets are explicitly listed in ...,direct_factual,"European Credit Card Fraud Detection Dataset, ...",True
4,Q005,Which optimization approach is used to tune XG...,direct_factual,Bayesian optimization of XGBoost hyperparameters.,True


,question_id,article_id,article_title,pdf_page_number,source_page_label,evidence_excerpt,verification_status
0,Q001,FD_003,Credit Card Fraud Detection Using AdaBoost and...,1,14277,"In this paper, a total of twelve machine learn...",verfied
1,Q001,FD_003,Credit Card Fraud Detection Using AdaBoost and...,7,14283,"A number of\nstandard models which include NB,...",verfied
2,Q002,FD_016,An integrated preprocessing and drift detectio...,1,92036,The method uses Mutual Information and SelectK...,verfied
3,Q003,FD_020,Hybrid feature selection framework for enhance...,1,1,"To address this, we propose a novel hybrid fea...",verified
4,Q003,FD_020,Hybrid feature selection framework for enhance...,29,29,"By leveraging SHAP, we provide a transparent a...",verified


Load your existing BM25 and FAISS indexes

In [11]:
# LOAD AND VERIFY FAISS AND BM25 INDEXES


# 1. JSONL LOADER

def read_jsonl(path):
    """
    Load metadata records from a JSONL file.
    Preserve the original record ordering.
    """

    if not path.is_file():
        raise FileNotFoundError(
            f"Metadata file not found: {path}"
        )

    with open(path, "r", encoding="utf-8") as file:
        records = [
            json.loads(line)
            for line in file
            if line.strip()
        ]

    if not records:
        raise ValueError(
            f"Empty metadata file: {path}"
        )

    return records


# 2. BM25 TOKENIZER

def bm25_tokenize(text):
    """
    Same tokenizer used when building the BM25 indexes.

    Convert text to lowercase and extract
    ASCII letters and digits.
    """

    return re.findall(
        r"[a-z0-9]+",
        str(text).lower()
    )


# 3. METADATA FINGERPRINT

def metadata_fingerprint(metadata):
    """
    Generate a SHA-256 fingerprint of the
    ordered metadata records.

    Uses:
    - vector_id
    - chunk_id
    - article_id
    - text
    """

    digest = hashlib.sha256()

    for record in metadata:

        line = (
            f"{int(record['vector_id'])}\t"
            f"{record['chunk_id']}\t"
            f"{record['article_id']}\t"
            f"{record['text']}\n"
        )

        digest.update(
            line.encode("utf-8")
        )

    return digest.hexdigest()


# 4. FIND BM25 MODEL IN SAVED PICKLE

def find_bm25_model(payload, depth=0):
    """
    Locate the BM25Okapi model inside a saved
    dictionary structure.
    """

    if isinstance(payload, BM25Okapi):
        return payload

    if isinstance(payload, dict) and depth < 3:

        possible_keys = [
            "bm25",
            "bm25_model",
            "model",
            "index",
            "retriever"
        ]

        for key in possible_keys:

            if key in payload:

                result = find_bm25_model(
                    payload[key],
                    depth + 1
                )

                if result is not None:
                    return result

    return None


# 5. PREPARE GOLD ARTICLE IDS

# Gold evidence was validated in the previous cell.

gold_article_ids = set(
    gold["article_id"]
    .dropna()
    .astype(str)
    .str.strip()
)

gold_article_ids.discard("")

if not gold_article_ids:
    raise ValueError(
        "No valid gold supporting articles found."
    )

print("Gold supporting articles:", len(gold_article_ids))


# 6. INITIALIZE STORAGE

indices = {}

metadata_by_strategy = {}

verification_summary = []

article_sets_by_strategy = {}

coverage_issues = []


# 7. LOAD AND VERIFY ALL INDEXES

for strategy in STRATEGIES:

    print("\n" + "=" * 60)
    print("STRATEGY:", strategy.upper())
    print("=" * 60)

    # 7.1 DEFINE FILE PATHS

    metadata_path = (
        EMBED_ROOT / strategy / "metadata.jsonl"
    )

    faiss_path = (
        FAISS_ROOT / strategy / "index.faiss"
    )

    bm25_path = (
        BM25_ROOT / strategy / "bm25_index.pkl"
    )

    for path in [
        metadata_path,
        faiss_path,
        bm25_path
    ]:

        if not path.is_file():
            raise FileNotFoundError(
                f"{strategy}: Missing file: {path}"
            )

    # 7.2 LOAD METADATA

    metadata = read_jsonl(metadata_path)

    required_fields = {
        "vector_id",
        "chunk_id",
        "article_id",
        "title",
        "text",
        "page_numbers"
    }

    for position, record in enumerate(metadata):

        missing = required_fields - set(record)

        if missing:
            raise ValueError(
                f"{strategy}: Metadata record {position} "
                f"is missing fields: {sorted(missing)}"
            )

        if (
            not isinstance(record["text"], str)
            or not record["text"].strip()
        ):
            raise ValueError(
                f"{strategy}: Invalid chunk text "
                f"at position {position}."
            )

        if (not isinstance(record["article_id"], str)
                or not record["article_id"].strip()):
            raise ValueError(
                f"{strategy}: Invalid article ID at position {position}."
            )

        if (not isinstance(record["chunk_id"], str)
                or not record["chunk_id"].strip()):
            raise ValueError(
                f"{strategy}: Invalid chunk ID at position {position}."
            )

    # 7.3 VALIDATE CHUNK IDS

    chunk_ids = [
        str(record["chunk_id"])
        for record in metadata
    ]

    if len(chunk_ids) != len(set(chunk_ids)):
        raise ValueError(
            f"{strategy}: Duplicate chunk IDs detected."
        )

    # 7.4 VALIDATE VECTOR IDS

    vector_ids = np.array(
        [
            int(record["vector_id"])
            for record in metadata
        ],
        dtype=np.int64
    )

    expected_ids = np.arange(
        len(metadata),
        dtype=np.int64
    )

    if not np.array_equal(vector_ids, expected_ids):
        raise ValueError(
            f"{strategy}: Vector IDs are not sequential "
            "or metadata ordering is incorrect."
        )

    # 7.5 LOAD FAISS INDEX

    faiss_index = faiss.read_index(
        str(faiss_path)
    )

    if not faiss_index.is_trained:
        raise ValueError(
            f"{strategy}: FAISS index is not trained."
        )

    # 7.6 LOAD BM25 INDEX

    # Only unpickle files created by your project.
    with open(bm25_path, "rb") as file:
        payload = pickle.load(file)

    if not isinstance(payload, dict):
        raise TypeError(
            f"{strategy}: Expected BM25 dictionary."
        )

    bm25_index = find_bm25_model(payload)

    if bm25_index is None:
        raise TypeError(
            f"{strategy}: BM25Okapi model not found."
        )

    # 7.7 VERIFY INDEX SIZES

    if faiss_index.ntotal != len(metadata):
        raise ValueError(
            f"{strategy}: FAISS metadata count mismatch."
        )

    if bm25_index.corpus_size != len(metadata):
        raise ValueError(
            f"{strategy}: BM25 metadata count mismatch."
        )

    # 7.8 VERIFY SAVED BM25 STRATEGY

    saved_strategy = payload.get("strategy")

    if saved_strategy != strategy:
        raise ValueError(
            f"{strategy}: BM25 strategy mismatch. "
            f"Found: {saved_strategy}"
        )

    # 7.9 VERIFY METADATA FINGERPRINT

    stored_fingerprint = payload.get(
        "fingerprint"
    )

    if not stored_fingerprint:
        raise ValueError(
            f"{strategy}: Missing BM25 fingerprint."
        )

    current_fingerprint = metadata_fingerprint(
        metadata
    )

    fingerprint_match = (
        stored_fingerprint == current_fingerprint
    )

    if not fingerprint_match:
        raise ValueError(
            f"{strategy}: BM25 metadata fingerprint "
            "does not match the current metadata."
        )

    # 7.10 VERIFY BM25 TOKEN FREQUENCIES

    if len(bm25_index.doc_freqs) != len(metadata):
        raise ValueError(
            f"{strategy}: BM25 frequency count mismatch."
        )

    if len(bm25_index.doc_len) != len(metadata):
        raise ValueError(
            f"{strategy}: BM25 document length mismatch."
        )

    mismatches = []

    for idx, record in enumerate(metadata):

        tokens = bm25_tokenize(
            record["text"]
        )

        expected_frequencies = Counter(tokens)

        stored_frequencies = Counter(
            bm25_index.doc_freqs[idx]
        )

        if expected_frequencies != stored_frequencies:
            mismatches.append(idx)
            continue

        if len(tokens) != bm25_index.doc_len[idx]:
            mismatches.append(idx)

    if mismatches:
        raise ValueError(
            f"{strategy}: {len(mismatches)} BM25 "
            "token mismatches detected. "
            f"First positions: {mismatches[:10]}"
        )

    # 7.11 CHECK GOLD ARTICLE COVERAGE

    indexed_article_ids = {
        str(record["article_id"]).strip()
        for record in metadata
    }

    article_sets_by_strategy[strategy] = (
        indexed_article_ids
    )

    missing_gold = (
        gold_article_ids - indexed_article_ids
    )

    if missing_gold:

        coverage_issues.append(
            (strategy, sorted(missing_gold))
        )

        print(
            "Missing gold articles:",
            sorted(missing_gold)
        )

    # 7.12 STORE VERIFIED INDEXES

    indices[strategy] = {
        "faiss": faiss_index,
        "bm25": bm25_index
    }

    metadata_by_strategy[strategy] = metadata

    # 7.13 STORE VERIFICATION SUMMARY

    verification_summary.append({

        "strategy": strategy,
        "total_chunks": len(metadata),
        "indexed_articles": len(indexed_article_ids),
        "faiss_count": faiss_index.ntotal,
        "bm25_count": bm25_index.corpus_size,
        "fingerprint_match": fingerprint_match,
        "token_mismatches": len(mismatches),
        "gold_articles_found": (
            len(gold_article_ids) - len(missing_gold)
        ),
        "missing_gold_articles": len(missing_gold),
        "status": (
            "VERIFIED" if not missing_gold
            else "MISSING_GOLD_ARTICLES"
        )

    })

    print("Total chunks:", len(metadata))
    print("Indexed articles:", len(indexed_article_ids))
    print("FAISS documents:", faiss_index.ntotal)
    print("BM25 documents:", bm25_index.corpus_size)
    print("Fingerprint match:", fingerprint_match)
    print("Token mismatches:", len(mismatches))

    print(
        "Gold articles found:",
        len(gold_article_ids) - len(missing_gold),
        "/",
        len(gold_article_ids)
    )


# 8. COMPARE ARTICLE COVERAGE ACROSS STRATEGIES

reference_strategy = STRATEGIES[0]

reference_articles = article_sets_by_strategy[
    reference_strategy
]

corpus_differences = []

for strategy in STRATEGIES[1:]:

    current_articles = article_sets_by_strategy[strategy]

    missing_from_strategy = (
        reference_articles - current_articles
    )

    extra_in_strategy = (
        current_articles - reference_articles
    )

    if missing_from_strategy or extra_in_strategy:

        corpus_differences.append(strategy)

        print(
            f"\nArticle coverage differs for {strategy}:"
        )

        print(
            "Missing relative to reference:",
            sorted(missing_from_strategy)
        )

        print(
            "Extra relative to reference:",
            sorted(extra_in_strategy)
        )


# 9. GENERATE VERIFICATION SUMMARY

verification_df = pd.DataFrame(
    verification_summary
)

indexed_articles = set().union(
    *article_sets_by_strategy.values()
)

print("\n" + "=" * 60)
print("INDEX ALIGNMENT VERIFICATION SUMMARY")
print("=" * 60)

display(verification_df)

print("\nTotal strategies:", len(indices))

print(
    "Total indexed chunks:",
    sum(
        len(metadata)
        for metadata in metadata_by_strategy.values()
    )
)

print(
    "Distinct indexed articles:",
    len(indexed_articles)
)

print(
    "Gold supporting articles:",
    len(gold_article_ids)
)


# 10. FINAL COVERAGE VALIDATION

if coverage_issues:

    details = [
        f"{strategy}: {missing}"
        for strategy, missing in coverage_issues
    ]

    raise ValueError(
        "Gold articles are missing from one or more "
        "chunking strategies:\n" + "\n".join(details)
    )

if corpus_differences:

    raise ValueError(
        "Indexed article sets differ across chunking "
        "strategies. Review coverage before comparing "
        f"retrieval methods: {corpus_differences}"
    )

print("\nAll gold articles are indexed in every strategy.")
print("All chunking strategies cover the same articles.")

print(
    "\nALL FAISS AND BM25 INDEXES VERIFIED SUCCESSFULLY."
)


Gold supporting articles: 83

STRATEGY: FIXED_SIZE
Total chunks: 5703
Indexed articles: 180
FAISS documents: 5703
BM25 documents: 5703
Fingerprint match: True
Token mismatches: 0
Gold articles found: 83 / 83

STRATEGY: SENTENCE_BASED
Total chunks: 5839
Indexed articles: 180
FAISS documents: 5839
BM25 documents: 5839
Fingerprint match: True
Token mismatches: 0
Gold articles found: 83 / 83

STRATEGY: RECURSIVE
Total chunks: 6106
Indexed articles: 180
FAISS documents: 6106
BM25 documents: 6106
Fingerprint match: True
Token mismatches: 0
Gold articles found: 83 / 83

STRATEGY: SECTION_AWARE
Total chunks: 7623
Indexed articles: 180
FAISS documents: 7623
BM25 documents: 7623
Fingerprint match: True
Token mismatches: 0
Gold articles found: 83 / 83

INDEX ALIGNMENT VERIFICATION SUMMARY


,strategy,total_chunks,indexed_articles,faiss_count,bm25_count,fingerprint_match,token_mismatches,gold_articles_found,missing_gold_articles,status
0,fixed_size,5703,180,5703,5703,True,0,83,0,VERIFIED
1,sentence_based,5839,180,5839,5839,True,0,83,0,VERIFIED
2,recursive,6106,180,6106,6106,True,0,83,0,VERIFIED
3,section_aware,7623,180,7623,7623,True,0,83,0,VERIFIED



Total strategies: 4
Total indexed chunks: 25271
Distinct indexed articles: 180
Gold supporting articles: 83

All gold articles are indexed in every strategy.
All chunking strategies cover the same articles.

ALL FAISS AND BM25 INDEXES VERIFIED SUCCESSFULLY.


Load the embedding model and Cross-Encoder

In [12]:
# LOAD MODELS

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

reranker = CrossEncoder(
    RERANKER_MODEL,
    max_length=512
)


# VALIDATE EMBEDDING DIMENSION

test_embedding = embedding_model.encode(
    ["embedding dimension check"],
    normalize_embeddings=True,
    convert_to_numpy=True
).astype("float32")


for strategy in STRATEGIES:

    expected_dimension = (
        indices[strategy]["faiss"].d
    )

    actual_dimension = test_embedding.shape[1]

    if actual_dimension != expected_dimension:

        raise ValueError(
            f"{strategy}: Embedding dimension mismatch."
        )


print("Embedding model loaded.")
print("Cross-Encoder loaded.")
print("All embedding dimensions match.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Embedding model loaded.
Cross-Encoder loaded.
All embedding dimensions match.


Implement the retrieval functions

In [13]:
# Reuse bm25_tokenize() from Cell 7 (index verification).


# 2. BM25 SEARCH

def bm25_search(query, strategy, k=FIRST_STAGE_K):
    """
    Retrieve the top-k passages using BM25.

    Returns:
        [(metadata_position, bm25_score), ...]
    """

    if k <= 0:
        raise ValueError("BM25 k must be positive.")

    bm25 = indices[strategy]["bm25"]

    tokens = bm25_tokenize(query)

    if not tokens:
        raise ValueError(
            "BM25 query contains no valid tokens."
        )

    # Calculate BM25 scores for all chunks
    scores = np.asarray(
        bm25.get_scores(tokens),
        dtype=np.float64
    )

    if not np.isfinite(scores).all():
        raise ValueError(
            f"{strategy}: Invalid BM25 scores detected."
        )

    # Sort scores from highest to lowest
    # Stable sorting preserves order for tied scores
    ranked_indices = np.argsort(
        -scores,
        kind="stable"
    )[:k]

    return [
        (int(idx), float(scores[idx]))
        for idx in ranked_indices
    ]


# 3. DENSE FAISS SEARCH

def dense_search(
    query_embedding,
    strategy,
    k=FIRST_STAGE_K
):
    """
    Retrieve top-k semantically similar passages
    using the existing FAISS index.

    Returns:
        [(metadata_position, faiss_score), ...]
    """

    if k <= 0:
        raise ValueError("FAISS k must be positive.")

    faiss_index = indices[strategy]["faiss"]

    # Convert query embedding to float32
    query_vector = np.asarray(
        query_embedding,
        dtype=np.float32
    ).reshape(1, -1)

    # Validate embedding dimension
    if query_vector.shape[1] != faiss_index.d:
        raise ValueError(
            f"{strategy}: Embedding dimension mismatch. "
            f"Expected {faiss_index.d}, "
            f"got {query_vector.shape[1]}."
        )

    if not np.isfinite(query_vector).all():
        raise ValueError(
            "Query embedding contains invalid values."
        )

    # Search the FAISS index
    scores, positions = faiss_index.search(
        query_vector,
        k
    )

    # Exclude invalid FAISS positions (-1)
    results = [
        (int(idx), float(score))
        for idx, score in zip(
            positions[0],
            scores[0]
        )
        if idx >= 0
    ]

    if any(
        not np.isfinite(score)
        for _, score in results
    ):
        raise ValueError(
            f"{strategy}: Invalid FAISS scores detected."
        )

    return results


# 4. RECIPROCAL RANK FUSION (RRF)

def reciprocal_rank_fusion(
    bm25_results,
    dense_results,
    k=RERANK_CANDIDATE_K,
    rrf_k=RRF_K
):
    """
    Combine BM25 and FAISS rankings using RRF.

    By default, retain 20 fused candidates
    for Cross-Encoder reranking.

    Returns:
        [(metadata_position, rrf_score), ...]
    """

    if k <= 0 or rrf_k <= 0:
        raise ValueError(
            "RRF k and rrf_k must be positive."
        )

    fused_scores = defaultdict(float)

    # Combine both retrieval rankings
    for results in [
        bm25_results,
        dense_results
    ]:

        for rank, (idx, _) in enumerate(
            results,
            start=1
        ):

            fused_scores[int(idx)] += (
                1.0 / (rrf_k + rank)
            )

    # Rank by combined RRF score
    # Resolve ties by metadata position
    ranked_indices = sorted(
        fused_scores,
        key=lambda idx: (
            -fused_scores[idx],
            idx
        )
    )[:k]

    return [
        (idx, float(fused_scores[idx]))
        for idx in ranked_indices
    ]


# 5. CROSS-ENCODER RERANKING

def cross_encoder_rerank(
    question,
    strategy,
    fused_results
):
    """
    Rerank fused candidate passages using
    the Cross-Encoder model.

    The function returns all supplied candidates
    in descending relevance score order.

    Final top-k selection is performed afterwards.
    """

    metadata = metadata_by_strategy[strategy]

    if not fused_results:
        return []

    # Construct question-passage pairs
    pairs = [
        (
            str(question),
            str(metadata[idx]["text"])
        )
        for idx, _ in fused_results
    ]

    # Predict relevance scores
    scores = reranker.predict(
        pairs,
        batch_size=RERANK_BATCH_SIZE,
        show_progress_bar=False
    )

    scores = np.asarray(
        scores,
        dtype=np.float64
    ).reshape(-1)

    # Validate prediction count
    if len(scores) != len(fused_results):
        raise ValueError(
            "Cross-Encoder returned an unexpected "
            "number of relevance scores."
        )

    if not np.isfinite(scores).all():
        raise ValueError(
            "Cross-Encoder returned invalid scores."
        )

    # Combine candidate positions and new scores
    reranked = [
        (int(idx), float(score))
        for (idx, _), score in zip(
            fused_results,
            scores
        )
    ]

    # Sort by descending relevance score
    reranked.sort(
        key=lambda item: (
            -item[1],
            item[0]
        )
    )

    return reranked


# 6. PAGE NUMBER NORMALIZATION

def ensure_pages(value):
    """
    Convert page metadata into a list
    of positive integer PDF page numbers.

    Supports:
    - Lists and tuples
    - NumPy arrays
    - JSON/Python-list strings
    - Single page numbers
    - Missing values
    """

    if value is None:
        return []

    # Handle lists and arrays
    if isinstance(value, (list, tuple, np.ndarray)):

        pages = []

        for page in value:
            pages.extend(ensure_pages(page))

        # Remove duplicates while preserving order
        return list(dict.fromkeys(pages))

    # Handle string values
    if isinstance(value, str):

        value = value.strip()

        if value.lower() in {
            "", "none", "null", "nan"
        }:
            return []

        try:
            decoded = json.loads(value)

        except json.JSONDecodeError:

            try:
                decoded = ast.literal_eval(value)

            except (ValueError, SyntaxError) as exc:
                raise ValueError(
                    f"Invalid page metadata: {value}"
                ) from exc

        return ensure_pages(decoded)

    # Handle missing numeric values
    if pd.isna(value):
        return []

    # Validate positive integer page numbers
    try:
        page_number = float(value)

    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"Invalid PDF page number: {value}"
        ) from exc

    if (
        not np.isfinite(page_number)
        or page_number < 1
        or not page_number.is_integer()
    ):
        raise ValueError(
            f"Invalid PDF page number: {value}"
        )

    return [int(page_number)]


# 7. FORMAT AND STORE RETRIEVED RESULTS

def add_ranked_rows(
    collector,
    question_row,
    strategy,
    method,
    hits
):
    """
    Convert retrieved passages into
    structured evaluation records.

    Store only the final top-10 passages.
    """

    metadata = metadata_by_strategy[strategy]

    for rank, (idx, score) in enumerate(
        hits[:FINAL_K],
        start=1
    ):

        # Validate metadata position
        if not 0 <= idx < len(metadata):
            raise IndexError(
                f"{strategy}: Invalid metadata position {idx}"
            )

        if not np.isfinite(score):
            raise ValueError(
                f"{strategy}: Invalid retrieval score."
            )

        record = metadata[idx]

        # Normalize PDF page numbers
        pages = ensure_pages(
            record.get("page_numbers", [])
        )

        collector.append({

            "question_id":
                str(question_row["question_id"]),

            "question_type":
                question_row["question_type"],

            "answerable":
                bool(question_row["answerable"]),

            "chunking_strategy":
                strategy,

            "method":
                method,

            "rank":
                rank,

            "chunk_id":
                str(record["chunk_id"]),

            "article_id":
                str(record["article_id"]),

            "article_title":
                str(record.get("title", "")),

            "pdf_page_numbers":
                json.dumps(pages),

            "score":
                float(score),

            "passage_text":
                str(record["text"])

        })


# 8. CONFIRM FUNCTION DEFINITIONS

print("=" * 60)
print("RETRIEVAL FUNCTIONS")
print("=" * 60)

print("BM25 search: READY")
print("Dense FAISS search: READY")
print("Reciprocal Rank Fusion: READY")
print("Cross-Encoder reranking: READY")
print("Page normalization: READY")
print("Result formatting: READY")

print("\nAll retrieval functions defined successfully.")


RETRIEVAL FUNCTIONS
BM25 search: READY
Dense FAISS search: READY
Reciprocal Rank Fusion: READY
Cross-Encoder reranking: READY
Page normalization: READY
Result formatting: READY

All retrieval functions defined successfully.


Run retrieval evaluation on all 80 questions


In [14]:
# RETRIEVAL EVALUATION
# BM25 | Dense FAISS | Hybrid RRF | Cross-Encoder Reranking


# 1. EVALUATION CONFIGURATION

METHODS = [
    "bm25",
    "dense_faiss",
    "hybrid_rrf",
    "hybrid_reranked"
]

# Validate required settings
assert FINAL_K > 0, "FINAL_K must be positive."
assert FIRST_STAGE_K >= RERANK_CANDIDATE_K >= FINAL_K, (
    "Require FIRST_STAGE_K >= RERANK_CANDIDATE_K >= FINAL_K."
)

assert len(questions) > 0, (
    "Evaluation question dataset is empty."
)

assert questions["question_id"].notna().all()
assert questions["question_id"].is_unique, (
    "Duplicate question IDs detected."
)

assert questions["question"].notna().all()

# Check that all indexes are available
for strategy in STRATEGIES:

    assert strategy in indices, (
        f"Missing index for {strategy}"
    )

    assert strategy in metadata_by_strategy, (
        f"Missing metadata for {strategy}"
    )


# 2. INITIALIZE RESULT COLLECTORS

results = []
timing_records = []

total_questions = len(questions)
total_strategies = len(STRATEGIES)
total_methods = len(METHODS)

expected_groups = (
    total_questions
    * total_strategies
    * total_methods
)

print("=" * 65)
print("STARTING RETRIEVAL EVALUATION")
print("=" * 65)

print("Research questions:", total_questions)
print("Chunking strategies:", total_strategies)
print("Retrieval methods:", total_methods)
print("Final top-K:", FINAL_K)
print("Expected configurations:", expected_groups)

print("\nMethods:")
for method in METHODS:
    print(" -", method)


# 3. EVALUATE ALL RESEARCH QUESTIONS

for question_row in tqdm(
    questions.to_dict("records"),
    desc="Evaluating research questions"
):

    question = str(
        question_row["question"]
    ).strip()

    qid = str(
        question_row["question_id"]
    )

    if not question:
        raise ValueError(
            f"Empty question text: {qid}"
        )


    # 3.1 GENERATE QUERY EMBEDDING

    start = perf_counter()

    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")[0]

    encoding_ms = (
        perf_counter() - start
    ) * 1000


    # 3.2 EVALUATE EACH CHUNKING STRATEGY

    for strategy in STRATEGIES:


        # A. BM25 RETRIEVAL

        start = perf_counter()

        bm25_results = bm25_search(
            question,
            strategy,
            k=FIRST_STAGE_K
        )

        bm25_ms = (
            perf_counter() - start
        ) * 1000


        # B. DENSE FAISS RETRIEVAL

        start = perf_counter()

        dense_results = dense_search(
            query_embedding,
            strategy,
            k=FIRST_STAGE_K
        )

        dense_ms = (
            perf_counter() - start
        ) * 1000


        # C. HYBRID RETRIEVAL USING RRF

        start = perf_counter()

        fused_candidates = reciprocal_rank_fusion(
            bm25_results,
            dense_results,
            k=RERANK_CANDIDATE_K,
            rrf_k=RRF_K
        )

        # Keep the top 10 RRF passages for the hybrid baseline.
        hybrid_results = fused_candidates[:FINAL_K]

        fusion_ms = (
            perf_counter() - start
        ) * 1000


        # D. CROSS-ENCODER RERANKING

        start = perf_counter()

        reranked_results = cross_encoder_rerank(
            question,
            strategy,
            fused_candidates
        )

        rerank_ms = (
            perf_counter() - start
        ) * 1000


        # 3.3 ORGANIZE RETRIEVAL RESULTS

        method_results = {

            "bm25":
                bm25_results[:FINAL_K],

            "dense_faiss":
                dense_results[:FINAL_K],

            "hybrid_rrf":
                hybrid_results[:FINAL_K],

            "hybrid_reranked":
                reranked_results[:FINAL_K]
        }


        # 3.4 CALCULATE METHOD LATENCIES

        method_times = {

            "bm25":
                bm25_ms,

            "dense_faiss":
                encoding_ms + dense_ms,

            "hybrid_rrf":
                (
                    bm25_ms
                    + encoding_ms
                    + dense_ms
                    + fusion_ms
                ),

            "hybrid_reranked":
                (
                    bm25_ms
                    + encoding_ms
                    + dense_ms
                    + fusion_ms
                    + rerank_ms
                )
        }


        # 3.5 VALIDATE METHOD OUTPUTS

        for method, hits in method_results.items():

            if len(hits) != FINAL_K:

                raise ValueError(
                    f"Question {qid}, strategy {strategy}, "
                    f"method {method}: Expected "
                    f"{FINAL_K} passages but got "
                    f"{len(hits)}."
                )

            # Check that retrieved positions are valid
            metadata_count = len(
                metadata_by_strategy[strategy]
            )

            retrieved_indices = [
                int(idx)
                for idx, score in hits
            ]

            if len(retrieved_indices) != len(
                set(retrieved_indices)
            ):
                raise ValueError(
                    f"Duplicate passage positions for "
                    f"{qid}, {strategy}, {method}."
                )

            for idx, score in hits:

                if not (0 <= int(idx) < metadata_count):
                    raise ValueError(
                        f"Invalid retrieved position: {idx}"
                    )

                if not np.isfinite(float(score)):
                    raise ValueError(
                        f"Non-finite retrieval score "
                        f"for {qid}, {strategy}, {method}."
                    )


            # 3.6 STORE RANKED PASSAGES

            add_ranked_rows(
                results,
                question_row,
                strategy,
                method,
                hits
            )


            # 3.7 STORE TIMING INFORMATION

            timing_records.append({

                "question_id":
                    qid,

                "chunking_strategy":
                    strategy,

                "method":
                    method,

                "retrieval_latency_ms":
                    round(
                        method_times[method],
                        3
                    ),

                "query_encoding_ms":
                    round(
                        encoding_ms,
                        3
                    )
            })


# 4. CONVERT RESULTS TO DATAFRAMES

results_df = pd.DataFrame(
    results
)

timing_df = pd.DataFrame(
    timing_records
)


# 5. VERIFY EVALUATION COMPLETENESS

group_columns = [
    "question_id",
    "chunking_strategy",
    "method"
]

actual_groups = (
    results_df.groupby(
        group_columns
    ).ngroups
)

if actual_groups != expected_groups:

    raise ValueError(
        f"Expected {expected_groups} configurations, "
        f"but found {actual_groups}."
    )


# 6. VALIDATE PASSAGE COUNTS

passage_counts = (
    results_df.groupby(
        group_columns
    ).size()
)

invalid_counts = passage_counts[
    passage_counts != FINAL_K
]

if not invalid_counts.empty:

    print("\nInvalid passage counts:")
    display(invalid_counts)

    raise ValueError(
        "Some configurations do not contain "
        f"exactly {FINAL_K} passages."
    )


# 7. VALIDATE RANK ORDER

for _, group in results_df.groupby(
    group_columns,
    sort=False
):

    actual_ranks = group["rank"].tolist()

    expected_ranks = list(
        range(1, FINAL_K + 1)
    )

    if actual_ranks != expected_ranks:

        raise ValueError(
            "Invalid passage ranking detected."
        )


# 8. VALIDATE RETRIEVAL METHODS

actual_methods = set(
    results_df["method"].unique()
)

if actual_methods != set(METHODS):

    raise ValueError(
        "Missing or unexpected retrieval methods. "
        f"Found: {actual_methods}"
    )

actual_strategies = set(
    results_df["chunking_strategy"].unique()
)

if actual_strategies != set(STRATEGIES):

    raise ValueError(
        "Missing or unexpected chunking strategies."
    )


# 9. VALIDATE TIMING RECORDS

if len(timing_df) != expected_groups:

    raise ValueError(
        "Timing record count does not match "
        "the expected evaluation configurations."
    )

if timing_df.duplicated(
    subset=group_columns
).any():

    raise ValueError(
        "Duplicate timing records detected."
    )

if not np.isfinite(
    timing_df["retrieval_latency_ms"]
).all():

    raise ValueError(
        "Invalid retrieval latency values detected."
    )


# 10. VALIDATE TOTAL RESULT COUNTS

expected_passages = (
    expected_groups * FINAL_K
)

if len(results_df) != expected_passages:

    raise ValueError(
        f"Expected {expected_passages} passage records, "
        f"but found {len(results_df)}."
    )


# 11. SAVE EVALUATION RESULTS
# RESULT_DIR was created and validated in Cell 3.

results_path = (
    RESULT_DIR /
    "retrieval_top10_all_methods.csv"
)

timing_path = (
    RESULT_DIR /
    "retrieval_timings.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

timing_df.to_csv(
    timing_path,
    index=False
)


# 12. RETRIEVAL EVALUATION SUMMARY

print("\n" + "=" * 65)
print("RETRIEVAL EVALUATION COMPLETED SUCCESSFULLY")
print("=" * 65)

print("Research questions:", total_questions)

print("Chunking strategies:", total_strategies)

print("Retrieval methods:", total_methods)

print("Expected configurations:", expected_groups)

print("Actual configurations:", actual_groups)

print("Expected ranked passages:", expected_passages)

print("Actual ranked passages:", len(results_df))

print("Timing records:", len(timing_df))


# 13. CONFIGURATION SUMMARY

configuration_summary = (
    results_df.groupby(
        ["chunking_strategy", "method"]
    ).agg(

        questions_evaluated=(
            "question_id",
            "nunique"
        ),

        retrieved_passages=(
            "chunk_id",
            "count"
        ),

        unique_articles=(
            "article_id",
            "nunique"
        )

    ).reset_index()
)

print("\nConfiguration summary:")

display(configuration_summary)


# 14. RETRIEVAL LATENCY SUMMARY

latency_summary = (
    timing_df.groupby(
        "method"
    )["retrieval_latency_ms"]
    .agg([
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
    .round(3)
)

print("\nRetrieval latency summary (milliseconds):")

display(latency_summary)


# 15. FINAL OUTPUT

print("\nResults saved successfully:")

print(results_path)

print(timing_path)

print("\nSample retrieval results:")

display(
    results_df.head(10)
)

print("\nAll retrieval evaluation checks PASSED.")


STARTING RETRIEVAL EVALUATION
Research questions: 95
Chunking strategies: 4
Retrieval methods: 4
Final top-K: 10
Expected configurations: 1520

Methods:
 - bm25
 - dense_faiss
 - hybrid_rrf
 - hybrid_reranked


Evaluating research questions:   0%|          | 0/95 [00:00<?, ?it/s]


RETRIEVAL EVALUATION COMPLETED SUCCESSFULLY
Research questions: 95
Chunking strategies: 4
Retrieval methods: 4
Expected configurations: 1520
Actual configurations: 1520
Expected ranked passages: 15200
Actual ranked passages: 15200
Timing records: 1520

Configuration summary:


,chunking_strategy,method,questions_evaluated,retrieved_passages,unique_articles
0,fixed_size,bm25,95,950,142
1,fixed_size,dense_faiss,95,950,146
2,fixed_size,hybrid_reranked,95,950,136
3,fixed_size,hybrid_rrf,95,950,143
4,recursive,bm25,95,950,143
5,recursive,dense_faiss,95,950,142
6,recursive,hybrid_reranked,95,950,141
7,recursive,hybrid_rrf,95,950,140
8,section_aware,bm25,95,950,146
9,section_aware,dense_faiss,95,950,146



Retrieval latency summary (milliseconds):


,mean,median,std,min,max
method,,,,,
bm25,58.316,46.622,50.371,22.968,433.807
dense_faiss,22.767,17.770,13.425,15.611,102.644
hybrid_reranked,214.074,185.766,104.567,150.302,871.609
hybrid_rrf,81.145,65.950,60.753,39.210,459.758



Results saved successfully:
/content/drive/MyDrive/NLP Project/data/retrieval_results/evaluation_95/retrieval_top10_all_methods.csv
/content/drive/MyDrive/NLP Project/data/retrieval_results/evaluation_95/retrieval_timings.csv

Sample retrieval results:


,question_id,question_type,answerable,chunking_strategy,method,rank,chunk_id,article_id,article_title,pdf_page_numbers,score,passage_text
0,Q001,direct_factual,True,fixed_size,bm25,1,fixed_size_00090,FD_003,Credit Card Fraud Detection Using AdaBoost and...,"[6, 7]",47.197442,GBT achieve a perfect fraud detection\nrate. T...
1,Q001,direct_factual,True,fixed_size,bm25,2,fixed_size_00077,FD_003,Credit Card Fraud Detection Using AdaBoost and...,[1],47.117357,"Received January 3, 2018, accepted February 10..."
2,Q001,direct_factual,True,fixed_size,bm25,3,fixed_size_04030,FD_140,An ensemble learning approach for anomaly dete...,[3],45.249958,did not handle the imbalanced data. Olowookere...
3,Q001,direct_factual,True,fixed_size,bm25,4,fixed_size_01020,FD_036,A soft voting ensemble learning approach for c...,"[3, 4]",45.049640,"Boost, Gradient Boosting, LightGBM, CatBoost, ..."
4,Q001,direct_factual,True,fixed_size,bm25,5,fixed_size_00089,FD_003,Credit Card Fraud Detection Using AdaBoost and...,"[5, 6]",44.865675,.\nA total of 11 features are used. The codes ...
5,Q001,direct_factual,True,fixed_size,bm25,6,fixed_size_00078,FD_003,Credit Card Fraud Detection Using AdaBoost and...,[1],44.785896,robustness of the algorithms. The experimental...
6,Q001,direct_factual,True,fixed_size,bm25,7,fixed_size_00614,FD_021,Enhancing credit card fraud detection using DB...,"[6, 7]",44.557303,note that the F1 score is particularly useful ...
7,Q001,direct_factual,True,fixed_size,bm25,8,fixed_size_00608,FD_021,Enhancing credit card fraud detection using DB...,[3],44.299508,was limited to insurance fraud detection.\nSap...
8,Q001,direct_factual,True,fixed_size,bm25,9,fixed_size_00079,FD_003,Credit Card Fraud Detection Using AdaBoost and...,"[1, 2]",44.128473,. There have\nbeen various studies on credit c...
9,Q001,direct_factual,True,fixed_size,bm25,10,fixed_size_01850,FD_065,Deep Learning for Credit Card Fraud Detection:...,"[2, 3]",43.421072,FRAUD DETECTION\nCredit card fraud occurs when...



All retrieval evaluation checks PASSED.


Generate the passage relevance annotation dataset

In [15]:
# PREPARE QUESTION AND GOLD LOOKUPS

question_lookup = (
    questions.set_index("question_id").to_dict("index")
)

gold_article_map = (
    gold.groupby("question_id")["article_id"]
    .agg(lambda values: set(values))
    .to_dict()
)

gold_page_map = defaultdict(set)

for row in gold.itertuples(index=False):

    gold_page_map[
        (str(row.question_id), str(row.article_id))
    ].add(int(row.pdf_page_number))


# CREATE UNIQUE PASSAGE POOL

keys = [
    "question_id",
    "chunking_strategy",
    "chunk_id"
]

pool_rows = []


for (qid, strategy, chunk_id), group in (
    results_df.groupby(keys, sort=False)
):

    sample = group.iloc[0]

    article_id = str(sample["article_id"])

    pages = ensure_pages(
        sample["pdf_page_numbers"]
    )

    question_gold_articles = gold_article_map.get(
        qid,
        set()
    )

    gold_article_match = (
        article_id in question_gold_articles
    )

    gold_page_overlap = bool(
        set(pages)
        & gold_page_map.get(
            (qid, article_id),
            set()
        )
    )


    # Methods that retrieved this chunk
    method_ranks = {
        row["method"]: int(row["rank"])
        for _, row in group.iterrows()
    }

    question_info = question_lookup[qid]


    pool_rows.append({

        "question_id":
            qid,

        "question_type":
            question_info["question_type"],

        "question":
            str(question_info["question"]),

        "reference_answer":
            str(question_info["reference_answer"])
            if question_info["answerable"]
            else "",

        "answerable":
            bool(question_info["answerable"]),

        "chunking_strategy":
            strategy,

        "chunk_id":
            chunk_id,

        "article_id":
            article_id,

        "article_title":
            sample["article_title"],

        "pdf_page_numbers":
            sample["pdf_page_numbers"],

        "retrieved_by":
            "; ".join(sorted(method_ranks)),

        "method_ranks":
            json.dumps(method_ranks),

        "gold_article_match":
            gold_article_match,

        "gold_page_overlap":
            gold_page_overlap,

        "passage_text":
            sample["passage_text"],

        # Manual annotation required
        "relevance":
            ""
    })


# CREATE DATAFRAME

pool_df = pd.DataFrame(pool_rows)

pool_df = pool_df.sort_values(
    [
        "question_id",
        "chunking_strategy",
        "gold_article_match",
        "gold_page_overlap"
    ],
    ascending=[
        True,
        True,
        False,
        False
    ]
).reset_index(drop=True)


# SAVE ANNOTATION TEMPLATE

JUDGMENTS_TEMPLATE = (
    RESULT_DIR / "passage_judgments_template_95.csv"
)

JUDGMENTS_FILE = (
    EVAL_DIR / "passage_judgments.csv"
)

pool_df.to_csv(
    JUDGMENTS_TEMPLATE,
    index=False
)


print("Annotation template created.")

print("Unique passages:", len(pool_df))
print("Template:", JUDGMENTS_TEMPLATE)

print("\nCreate the reviewed file at:")
print(JUDGMENTS_FILE)

display(
    pool_df[
        [
            "question_id",
            "chunking_strategy",
            "article_id",
            "chunk_id",
            "gold_article_match",
            "relevance"
        ]
    ].head(10)
)


Annotation template created.
Unique passages: 7096
Template: /content/drive/MyDrive/NLP Project/data/retrieval_results/evaluation_95/passage_judgments_template_95.csv

Create the reviewed file at:
/content/drive/MyDrive/NLP Project/data/evaluation/passage_judgments.csv


,question_id,chunking_strategy,article_id,chunk_id,gold_article_match,relevance
0,Q001,fixed_size,FD_003,fixed_size_00090,True,
1,Q001,fixed_size,FD_003,fixed_size_00077,True,
2,Q001,fixed_size,FD_003,fixed_size_00078,True,
3,Q001,fixed_size,FD_003,fixed_size_00079,True,
4,Q001,fixed_size,FD_003,fixed_size_00089,True,
5,Q001,fixed_size,FD_140,fixed_size_04030,False,
6,Q001,fixed_size,FD_036,fixed_size_01020,False,
7,Q001,fixed_size,FD_021,fixed_size_00614,False,
8,Q001,fixed_size,FD_021,fixed_size_00608,False,
9,Q001,fixed_size,FD_065,fixed_size_01850,False,


Calculate provisional article coverage and retrieval latency

In [16]:
# ARTICLE-LEVEL AND PAGE-LEVEL RETRIEVAL EVALUATION

K_VALUES = [1, 3, 5, 10]


# 1. REUSE VALIDATED GOLD EVIDENCE LOOKUPS
# gold_article_map and gold_page_map were created in Cell 15.
# Reusing them avoids loading and grouping the same gold evidence twice.
if not gold_article_map or not gold_page_map:
    raise ValueError("Run Cell 15 to construct gold evidence lookups.")


# 3. PREPARE RETRIEVAL RESULTS

retrieved = results_df.copy()

retrieved["question_id"] = (
    retrieved["question_id"].astype(str).str.strip()
)

retrieved["article_id"] = (
    retrieved["article_id"].astype(str).str.strip()
)

retrieved["rank"] = (
    pd.to_numeric(
        retrieved["rank"],
        errors="raise"
    ).astype(int)
)


# 4. PAGE NUMBER NORMALIZATION
# Reuse ensure_pages() from Cell 11.


# 5. CALCULATE nDCG FOR BINARY ARTICLE RELEVANCE

def binary_ndcg(relevance, total_gold, k):

    relevance = np.asarray(
        relevance[:k],
        dtype=float
    )

    if total_gold == 0:
        return 0.0

    positions = np.arange(
        2,
        len(relevance) + 2
    )

    dcg = np.sum(
        relevance / np.log2(positions)
    )

    ideal_count = min(
        total_gold,
        k
    )

    ideal_positions = np.arange(
        2,
        ideal_count + 2
    )

    idcg = np.sum(
        1.0 / np.log2(ideal_positions)
    )

    return float(dcg / idcg) if idcg > 0 else 0.0


# 6. CALCULATE METRICS FOR EVERY CONFIGURATION

metric_rows = []

answerable_set = {
    str(qid).strip()
    for qid in answerable_ids
}

for (qid, strategy, method), group in retrieved.groupby(
    [
        "question_id",
        "chunking_strategy",
        "method"
    ]
):

    if qid not in answerable_set:
        continue

    group = group.sort_values(
        "rank"
    )

    known_articles = gold_article_map.get(qid, set())

    if not known_articles:
        raise ValueError(
            f"No gold articles for answerable question {qid}"
        )

    # Create unique article ranking
    # Preserve first appearance order

    unique_articles = list(
        dict.fromkeys(
            group["article_id"].tolist()
        )
    )

    article_relevance = [
        int(article in known_articles)
        for article in unique_articles
    ]

    row_metrics = {
        "question_id": qid,
        "chunking_strategy": strategy,
        "method": method,
        "gold_article_count": len(known_articles),
        "unique_articles_in_top10_passages":
            len(unique_articles)
    }


    # 7. COMPUTE METRICS AT EACH K

    for k in K_VALUES:

        # A. ARTICLE COVERAGE IN TOP-K PASSAGES

        top_passages = group.head(k)

        retrieved_article_set = set(
            top_passages["article_id"]
        )

        matched_articles = (
            retrieved_article_set & known_articles
        )

        row_metrics[
            f"article_hit_at_{k}_passages"
        ] = int(bool(matched_articles))

        row_metrics[
            f"article_coverage_at_{k}_passages"
        ] = (
            len(matched_articles)
            / len(known_articles)
        )


        # B. UNIQUE ARTICLE PRECISION AND RECALL

        top_article_relevance = (
            article_relevance[:k]
        )

        relevant_count = sum(
            top_article_relevance
        )

        row_metrics[
            f"article_precision_at_{k}"
        ] = relevant_count / k

        row_metrics[
            f"article_recall_at_{k}"
        ] = (
            relevant_count / len(known_articles)
        )


        # C. ARTICLE MRR@K

        relevant_positions = [
            index + 1
            for index, relevance
            in enumerate(top_article_relevance)
            if relevance == 1
        ]

        if relevant_positions:

            mrr = (
                1.0 / relevant_positions[0]
            )

        else:
            mrr = 0.0

        row_metrics[
            f"article_mrr_at_{k}"
        ] = mrr


        # D. ARTICLE nDCG@K

        row_metrics[
            f"article_ndcg_at_{k}"
        ] = binary_ndcg(
            article_relevance,
            len(known_articles),
            k
        )


        # E. PAGE OVERLAP HIT@K

        page_match = False

        for passage in top_passages.itertuples():

            article_id = passage.article_id

            pages = ensure_pages(
                passage.pdf_page_numbers
            )

            if any(
                page in gold_page_map.get((qid, article_id), set())
                for page in pages
            ):
                page_match = True
                break

        row_metrics[
            f"page_overlap_hit_at_{k}"
        ] = int(page_match)


    metric_rows.append(
        row_metrics
    )


# 8. CONVERT METRICS TO DATAFRAME

article_metrics_df = pd.DataFrame(
    metric_rows
)


# 9. VALIDATE EVALUATION COVERAGE

expected_rows = (
    len(answerable_set)
    * len(STRATEGIES)
    * 4
)

if len(article_metrics_df) != expected_rows:

    raise ValueError(
        f"Expected {expected_rows} metric rows, "
        f"found {len(article_metrics_df)}."
    )


# 10. CALCULATE SUMMARY BY STRATEGY AND METHOD

metric_columns = [
    column
    for column in article_metrics_df.columns
    if (
        column.startswith("article_")
        or column.startswith("page_overlap_")
    )
    and column not in [
        "article_id",
        "gold_article_count"
    ]
]

summary_df = (
    article_metrics_df.groupby(
        [
            "chunking_strategy",
            "method"
        ],
        as_index=False
    )[metric_columns]
    .mean()
)


# 11. SAVE METRIC RESULTS
# RESULT_DIR was already created in Cell 3.

per_question_path = (
    RESULT_DIR
    / "article_page_metrics_per_question.csv"
)

summary_path = (
    RESULT_DIR
    / "article_page_metrics_summary.csv"
)

article_metrics_df.to_csv(
    per_question_path,
    index=False
)

summary_df.to_csv(
    summary_path,
    index=False
)


# 12. DISPLAY FINAL SUMMARY

print("=" * 65)
print("ARTICLE-LEVEL AND PAGE-LEVEL EVALUATION COMPLETED")
print("=" * 65)

print("Answerable questions:", len(answerable_set))
print("Chunking strategies:", len(STRATEGIES))
print("Methods:", 4)

print(
    "Question-method-strategy evaluations:",
    len(article_metrics_df)
)

print("\nKey metrics at K=5:")

display(
    summary_df[
        [
            "chunking_strategy",
            "method",
            "article_hit_at_5_passages",
            "article_coverage_at_5_passages",
            "article_precision_at_5",
            "article_recall_at_5",
            "article_mrr_at_5",
            "article_ndcg_at_5",
            "page_overlap_hit_at_5"
        ]
    ].round(4)
)

print("\nSaved files:")
print(per_question_path)
print(summary_path)



ARTICLE-LEVEL AND PAGE-LEVEL EVALUATION COMPLETED
Answerable questions: 70
Chunking strategies: 4
Methods: 4
Question-method-strategy evaluations: 1120

Key metrics at K=5:


,chunking_strategy,method,article_hit_at_5_passages,article_coverage_at_5_passages,article_precision_at_5,article_recall_at_5,article_mrr_at_5,article_ndcg_at_5,page_overlap_hit_at_5
0,fixed_size,bm25,0.9571,0.7605,0.2857,0.8250,0.8724,0.7945,0.8429
1,fixed_size,dense_faiss,0.9429,0.7095,0.2514,0.7683,0.7893,0.7134,0.7571
2,fixed_size,hybrid_reranked,0.9714,0.7781,0.2914,0.8243,0.8917,0.8144,0.8571
3,fixed_size,hybrid_rrf,0.9857,0.7743,0.2857,0.8212,0.8905,0.8072,0.8429
4,recursive,bm25,0.9571,0.7750,0.2829,0.8190,0.8683,0.7995,0.8429
5,recursive,dense_faiss,0.9429,0.7264,0.2657,0.7817,0.7802,0.7130,0.7143
6,recursive,hybrid_reranked,0.9714,0.7871,0.2914,0.8267,0.8869,0.8189,0.8429
7,recursive,hybrid_rrf,0.9571,0.7712,0.2943,0.8395,0.8898,0.8169,0.8143
8,section_aware,bm25,0.9286,0.7557,0.2800,0.8193,0.8350,0.7749,0.8857
9,section_aware,dense_faiss,0.9571,0.7252,0.2600,0.7579,0.8024,0.7023,0.7429



Saved files:
/content/drive/MyDrive/NLP Project/data/retrieval_results/evaluation_95/article_page_metrics_per_question.csv
/content/drive/MyDrive/NLP Project/data/retrieval_results/evaluation_95/article_page_metrics_summary.csv


RETRIEVAL FAILURE ANALYSIS

In [17]:

# RETRIEVAL FAILURE ANALYSIS

metrics_file = (
    RESULT_DIR / "article_page_metrics_per_question.csv"
)

metrics_df = pd.read_csv(metrics_file)

# Select the proposed best retrieval configuration
selected_results = metrics_df[
    (metrics_df["chunking_strategy"] == "section_aware")
    & (metrics_df["method"] == "hybrid_reranked")
].copy()

# Identify failures at K=5
failed_questions = selected_results[
    selected_results["page_overlap_hit_at_5"] == 0
].copy()

print("Total evaluated questions:", len(selected_results))
print("Questions without page overlap:", len(failed_questions))

# Inspect failed questions
display(failed_questions)

# Save for manual review
failure_file = (
    RESULT_DIR / "retrieval_failure_analysis.csv"
)

failed_questions.to_csv(
    failure_file,
    index=False
)

print("\nFailure analysis saved to:", failure_file)


Total evaluated questions: 70
Questions without page overlap: 6


,question_id,chunking_strategy,method,gold_article_count,unique_articles_in_top10_passages,article_hit_at_1_passages,article_coverage_at_1_passages,article_precision_at_1,article_recall_at_1,article_mrr_at_1,...,article_mrr_at_5,article_ndcg_at_5,page_overlap_hit_at_5,article_hit_at_10_passages,article_coverage_at_10_passages,article_precision_at_10,article_recall_at_10,article_mrr_at_10,article_ndcg_at_10,page_overlap_hit_at_10
602,Q038,section_aware,hybrid_reranked,4,7,0,0.0,0.0,0.0,0.0,...,0.5,0.397322,0,1,0.75,0.3,0.75,0.5,0.536378,1
682,Q043,section_aware,hybrid_reranked,1,3,0,0.0,0.0,0.0,0.0,...,0.5,0.630930,0,1,1.00,0.1,1.00,0.5,0.630930,1
794,Q050,section_aware,hybrid_reranked,2,2,1,0.5,1.0,0.5,1.0,...,1.0,1.000000,0,1,1.00,0.2,1.00,1.0,1.000000,1
922,Q058,section_aware,hybrid_reranked,2,7,1,0.5,1.0,0.5,1.0,...,1.0,0.613147,0,1,0.50,0.1,0.50,1.0,0.613147,0
1034,Q065,section_aware,hybrid_reranked,2,9,0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0,0,0.00,0.0,0.00,0.0,0.000000,0
1082,Q068,section_aware,hybrid_reranked,2,6,0,0.0,0.0,0.0,0.0,...,0.5,0.624051,0,1,1.00,0.2,1.00,0.5,0.624051,0



Failure analysis saved to: /content/drive/MyDrive/NLP Project/data/retrieval_results/evaluation_95/retrieval_failure_analysis.csv
